# ArchiMind dataset processing (Colab driver)

Setup guide: `docs/developer/DATASET_COLAB.md`. This notebook only calls the tested
`python -m archimind.datasets` commands. Edit **only** the configuration cell.
Re-running from the top with the same `RUN_ID` resumes: finished stages are skipped.

In [ ]:
# --- Configuration (edit here only) ---
RUN_ID = "pilot-001"  # reuse the same id to resume after a disconnect
ARCHIMIND_COMMIT = (
    "75f7dba6ff73d78bee6eb4f8d0d0caf440dbaaac"  # exact commit, never a branch
)
REPO = "ngstephen1/ArchiMind"
WORK = "/content/work"  # fast local disk (lost when the runtime ends)
DRIVE_ROOT = "/content/drive/MyDrive/ArchiMind-data"  # persistent backup
FLOOR_GIB = 4  # stop below this much free local disk

In [ ]:
# Mount Drive and create the run layout.
import datetime
import os
import pathlib
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

RUN_DRIVE = pathlib.Path(DRIVE_ROOT) / "runs" / RUN_ID
for d in ("raw", "stages", "packs", "receipts"):
    (pathlib.Path(WORK) / d).mkdir(parents=True, exist_ok=True)
for d in ("stages", "packs", "receipts"):
    (RUN_DRIVE / d).mkdir(parents=True, exist_ok=True)
print("local work:", WORK)
print("drive backup:", RUN_DRIVE)

In [ ]:
# Load secrets into this process only. Values are never printed.
from google.colab import userdata

for name in (
    "GH_TOKEN",
    "HF_TOKEN",
    "KAGGLE_USERNAME",
    "KAGGLE_KEY",
    "KAGGLE_API_TOKEN",
):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: loaded")
    except Exception:
        print(
            f"{name}: not available (add it under the key icon and enable notebook access)"
        )

In [ ]:
# Install the pinned ArchiMind commit. Git output is hidden so the token never appears.
import base64
import re

assert re.fullmatch(r"[0-9a-f]{40}", ARCHIMIND_COMMIT), (
    "ARCHIMIND_COMMIT must be an exact 40-character commit"
)
SRC = "/content/ArchiMind"
PLAIN = f"https://github.com/{REPO}.git"


def _git(*args, cwd=SRC):
    env = os.environ.copy()
    token = env.get("GH_TOKEN", "")
    if token:
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}",
        )
    r = subprocess.run(["git", *args], cwd=cwd, env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(
            f"git {args[0]} failed; output hidden to protect the token. Check GH_TOKEN."
        )


if not os.path.isdir(f"{SRC}/.git"):
    _git("clone", "--quiet", "--no-checkout", PLAIN, SRC, cwd="/content")
try:
    _git("fetch", "--quiet", "origin", ARCHIMIND_COMMIT)
    _git("checkout", "--quiet", "--detach", ARCHIMIND_COMMIT)
finally:
    _git("remote", "set-url", "origin", PLAIN)  # keep origin credential-free

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", SRC],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.14.1"], check=True
)
os.environ["ARCHIMIND_COMMIT"] = ARCHIMIND_COMMIT
print("installed", ARCHIMIND_COMMIT)

In [ ]:
# Helpers: run a dataset command, checkpoint to Drive, restore from Drive.
def ds(*args):
    r = subprocess.run(
        [sys.executable, "-m", "archimind.datasets", *map(str, args)],
        capture_output=True,
        text=True,
    )
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"archimind.datasets {args[0]} exited {r.returncode}")


def checkpoint():
    for d in ("stages", "packs", "receipts"):
        ds("sync", f"{WORK}/{d}", RUN_DRIVE / d)


def restore():
    for d in ("stages", "packs", "receipts"):
        ds("sync", RUN_DRIVE / d, f"{WORK}/{d}")

In [ ]:
# Restore earlier progress for this RUN_ID (no-op on a fresh run).
restore()

In [ ]:
# Environment receipt and free-space check (fails below FLOOR_GIB).
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ds(
    "env",
    "--data-root",
    WORK,
    "--floor-gib",
    FLOOR_GIB,
    "--out",
    f"{WORK}/receipts/env-{stamp}.json",
)
checkpoint()

In [ ]:
# Self-check: build and verify a synthetic pack end to end, then checkpoint it.
ds("fixture-pack", f"{WORK}/packs/synthetic-selfcheck")
ds("verify-pack", f"{WORK}/packs/synthetic-selfcheck")
checkpoint()

## P1: probe (inventory and yield)

Outputs go to `WORK/stages/p1` and are checkpointed to Drive. SSR and MIDI pins
come from the P1 probe of 2026-11-06. Re-running skips files that already match
their pinned SHA-256. Only the FUTURE cell needs Kaggle credentials.

In [ ]:
# SSR-3DFRONT: pinned revision; ~7.4 MB of parquet in total.
P1 = f"{WORK}/stages/p1"
SSR_REPO = "gradient-spaces/SSR-3DFRONT"
SSR_REV = "b606183e56d972f85850125f9191932cbea654c3"
SSR_SHARDS = {  # path -> SHA-256 published by Hugging Face (LFS oid)
    "data/train-00000-of-00001.parquet": "aedbb1f9448614299ecc6c40e6ccf4f9b7441feee42275d7fcfb06690de038e1",
    "data/val-00000-of-00001.parquet": "38e99958698c2633199e041f68cba51b91f2db1255718ea894a6db8ee323fe40",
    "data/test-00000-of-00001.parquet": "8970830a27fb3dafb5e89d324f5f3666b3defd489e79345d314d455d00a1331b",
}
HF = "https://huggingface.co/datasets"
ds("probe-hf", SSR_REPO, "--revision", SSR_REV, "--out", f"{P1}/ssr_inventory.json")
local = []
for path, sha in SSR_SHARDS.items():
    dest = f"{WORK}/raw/ssr/{path.rsplit('/', 1)[-1]}"
    ds(
        "fetch",
        f"{HF}/{SSR_REPO}/resolve/{SSR_REV}/{path}",
        dest,
        "--max-bytes",
        8_000_000,
        "--sha256",
        sha,
    )
    local.append(dest)
ds("probe-ssr", *local, "--out", f"{P1}/ssr_probe.json")
checkpoint()

In [ ]:
# MIDI-3D: ID lists (a few MB) plus a streamed 1 GiB prefix listing of the test-scene
# archive. The prefix is read and discarded; nothing from it is stored.
MIDI_REPO = "huanngzh/3D-Front"
MIDI_REV = "156fc5312ea613d4e92cbf5da1fddfa61bbad3b2"
MIDI = f"{HF}/{MIDI_REPO}/resolve/{MIDI_REV}"
ds("probe-hf", MIDI_REPO, "--revision", MIDI_REV, "--out", f"{P1}/midi_inventory.json")
for name in (
    "midi_room_ids.json",
    "midi_furniture_ids.json",
    "midi_test_room_ids.json",
    "midi_test_furniture_ids.json",
):
    ds("fetch", f"{MIDI}/{name}", f"{P1}/midi/{name}", "--max-bytes", 5_000_000)
ds(
    "probe-tar-prefix",
    f"{MIDI}/3D-FRONT-TEST-SCENE.tar.gz",
    "--max-bytes",
    1024**3,
    "--out",
    f"{P1}/midi_test_scene_prefix.json",
)
checkpoint()

In [ ]:
# 3D-FUTURE (Kaggle mirror): version, full file list and model_info.json, the
# authoritative jid -> category table. Needs KAGGLE_USERNAME and KAGGLE_KEY.
import json

FUTURE_REF = "tobetheonly/3d-future-model"
ds("probe-kaggle", FUTURE_REF, "--out", f"{P1}/future_inventory.json")
inventory = json.load(open(f"{P1}/future_inventory.json"))
print("model_info candidates:", inventory.get("model_info"))
for i, entry in enumerate(inventory.get("model_info") or []):
    ds(
        "fetch",
        entry["url"],
        f"{P1}/future/model_info_{i}.download",
        "--max-bytes",
        64 * 1024**2,
        "--kaggle-auth",
    )
checkpoint()

**Hand-back:** download `MyDrive/ArchiMind-data/runs/<RUN_ID>/stages/p1/` and place
it at `E:rchimind-data\p1\` (outside the Git checkout). P2 reads it from there.

In [ ]:
# Finalize before closing the tab: checkpoint, then make Drive finish writing.
checkpoint()
drive.flush_and_unmount()
print("Drive flushed. Safe to close.")